# Student Project: End-to-End House Price Prediction
## Phase 2 — Exploratory Data Analysis, Cleaning, & Linear Regression Model Pipeline

This notebook implements the complete machine learning workflow for predicting house prices in India:
1. **Load & Inspect**: Data ingestion, structure, and missingness audit.
2. **Exploratory Data Analysis (EDA)**: Visualizing distributions, correlations, and categorical patterns.
3. **Data Cleaning & Feature Engineering**: Parsing unstructured price strings (`Lac`/`Cr`), normalizing area units (`sqm` to `sqft`), parsing floor levels, grouping high-cardinality locations, and removing outliers.
4. **Pipeline & Modeling**: Building an end-to-end `ColumnTransformer` + `LinearRegression` Scikit-Learn pipeline.
5. **Evaluation**: Reporting MAE, RMSE, $R^2$ on the unseen test set and visualizing residuals.
6. **Artifact Export**: Exporting `house_price.pkl` and `locations.json` for production deployment.

In [ ]:
import os
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

---
## 2.1 Load & Inspect Dataset

In [ ]:
# Load dataset from data folder (with fallback paths)
data_paths = [Path("data/house_prices.csv"), Path("notebooks/data/house_prices.csv")]
csv_path = next((p for p in data_paths if p.exists()), Path("data/house_prices.csv"))

df = pd.read_csv(csv_path, low_memory=False)
print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns\n")
df.head()

In [ ]:
# Inspect column data types and null counts
df.info()

In [ ]:
# Missing values percentage per column
missing_summary = df.isna().mean().sort_values(ascending=False) * 100
print("Percentage of Missing Values Per Column:")
print(missing_summary.round(2).to_string())

### Initial Data Assessment:
- **Total Volume**: Contains real property listings from India (~187,000 raw rows).
- **Types**: Most fields (`Amount(in rupees)`, `Carpet Area`, `Floor`, `Furnishing`) are text/object strings requiring regex normalization.
- **Missingness**: Sparse columns such as `Dimensions`, `Plot Area`, and `overlooking` have substantial missing rates and will be excluded, while key attributes (`Bathroom`, `Balcony`, `Furnishing`) will be imputed in the preprocessing pipeline.

---
## 2.2 Exploratory Data Analysis (EDA)

In [ ]:
# Pre-parse target price for EDA visualization
def quick_parse_price(x):
    if pd.isna(x):
        return None
    s = str(x).lower().replace(",", "")
    if "call for price" in s:
        return None
    try:
        if "cr" in s:
            return float(re.findall(r"[-+]?(?:\d*\.\d+|\d+)", s)[0]) * 1e7
        if "lac" in s or "lakh" in s:
            return float(re.findall(r"[-+]?(?:\d*\.\d+|\d+)", s)[0]) * 1e5
        nums = re.findall(r"[-+]?(?:\d*\.\d+|\d+)", s)
        return float(nums[0]) if nums else None
    except:
        return None

df["price_clean"] = df["Amount(in rupees)"].apply(quick_parse_price)
eda_df = df.dropna(subset=["price_clean"]).copy()

In [ ]:
# Plot 1: Target Price Distribution (Log Scale)
plt.figure(figsize=(10, 5))
sns.histplot(eda_df["price_clean"], kde=True, log_scale=True, color="#2563eb")
plt.title("Plot 1: Distribution of Property Prices (Log Scale)", fontsize=14, fontweight="bold")
plt.xlabel("Price in INR (Log Scale)")
plt.ylabel("Listing Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Price vs Carpet Area
def quick_parse_area(x):
    if pd.isna(x):
        return None
    s = str(x).lower().replace(",", "")
    nums = re.findall(r"[-+]?(?:\d*\.\d+|\d+)", s)
    if not nums:
        return None
    val = float(nums[0])
    return val * 10.764 if ("sqm" in s or "sq.m" in s) else val

eda_df["carpet_area_sqft"] = eda_df["Carpet Area"].apply(quick_parse_area)
sample_scatter = eda_df.dropna(subset=["carpet_area_sqft", "price_clean"]).sample(min(2000, len(eda_df)), random_state=42)

plt.figure(figsize=(10, 5))
sns.scatterplot(data=sample_scatter, x="carpet_area_sqft", y="price_clean", alpha=0.6, color="#7c3aed")
plt.title("Plot 2: Price vs. Carpet Area (sq. ft.)", fontsize=14, fontweight="bold")
plt.xlabel("Carpet Area (sqft)")
plt.ylabel("Price (INR)")
plt.yscale("log")
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Average Price by Top 15 Locations
top_15_locs = eda_df["location"].value_counts().head(15).index
avg_price_loc = eda_df[eda_df["location"].isin(top_15_locs)].groupby("location")["price_clean"].mean().sort_values(ascending=True)

plt.figure(figsize=(10, 6))
avg_price_loc.plot(kind="barh", color="#0d9488")
plt.title("Plot 3: Average Property Valuation Across Top 15 Locations", fontsize=14, fontweight="bold")
plt.xlabel("Average Price (INR)")
plt.ylabel("Location / City Region")
plt.tight_layout()
plt.show()

In [ ]:
# Plot 4: Price by Furnishing Status & Number of Bathrooms
eda_df["bath_num"] = pd.to_numeric(eda_df["Bathroom"], errors="coerce").fillna(2).astype(int)
sample_box = eda_df[eda_df["bath_num"].between(1, 4)].dropna(subset=["Furnishing"]).copy()

plt.figure(figsize=(10, 5))
sns.boxplot(data=sample_box, x="Furnishing", y="price_clean", hue="bath_num", palette="Blues")
plt.yscale("log")
plt.title("Plot 4: Price by Furnishing Status & Bathroom Count", fontsize=14, fontweight="bold")
plt.xlabel("Furnishing Level")
plt.ylabel("Price (INR, Log Scale)")
plt.legend(title="Bathrooms", loc="upper right")
plt.tight_layout()
plt.show()

### EDA Key Findings:
1. **Right-Skewed Target Distribution**: Property prices span multiple orders of magnitude (from lakhs to multi-crores). Training on log-transformed target `np.log1p(y)` and evaluating via `np.expm1` stabilizes gradient updates and reduces variance.
2. **Area & Bathroom Correlation**: Carpet area and bathroom count exhibit strong positive monotonic relationships with property price.
3. **Locality Premium**: Prime metropolitan locations command significant multiples over regional suburbs, justifying one-hot encoded top location indicators.

---
## 2.3 Cleaning & Feature Engineering

In [ ]:
def parse_amount(x):
    if pd.isna(x):
        return None
    s = str(x).strip().lower().replace(",", "")
    if "call for price" in s or "price on request" in s:
        return None
    try:
        if "cr" in s:
            return float(re.findall(r"[-+]?(?:\d*\.\d+|\d+)", s)[0]) * 1e7
        elif "lac" in s or "lakh" in s:
            return float(re.findall(r"[-+]?(?:\d*\.\d+|\d+)", s)[0]) * 1e5
        nums = re.findall(r"[-+]?(?:\d*\.\d+|\d+)", s)
        return float(nums[0]) if nums else None
    except:
        return None

def parse_area(x):
    if pd.isna(x):
        return None
    s = str(x).strip().lower().replace(",", "")
    nums = re.findall(r"[-+]?(?:\d*\.\d+|\d+)", s)
    if not nums:
        return None
    val = float(nums[0])
    if "sqm" in s or "sq.m" in s:
        return round(val * 10.764, 2)
    return round(val, 2)

def parse_floor(x):
    if pd.isna(x):
        return 1
    s = str(x).strip().lower()
    if "lower basement" in s:
        return -2
    if "basement" in s:
        return -1
    if "ground" in s:
        return 0
    nums = re.findall(r"[-+]?\d+", s)
    return int(nums[0]) if nums else 1

# Apply parsers
clean_df = df.copy()
clean_df["price_clean"] = clean_df["Amount(in rupees)"].apply(parse_amount)
clean_df = clean_df.dropna(subset=["price_clean"])
clean_df = clean_df[clean_df["price_clean"] > 0]

clean_df["carpet_area_sqft"] = clean_df["Carpet Area"].apply(parse_area)
if "Super Area" in clean_df.columns:
    clean_df["carpet_area_sqft"] = clean_df["carpet_area_sqft"].fillna(clean_df["Super Area"].apply(parse_area))
clean_df = clean_df.dropna(subset=["carpet_area_sqft"])
clean_df = clean_df[(clean_df["carpet_area_sqft"] >= 100) & (clean_df["carpet_area_sqft"] <= 50000)]

clean_df["floor_num"] = clean_df["Floor"].apply(parse_floor)
clean_df["bathroom"] = pd.to_numeric(clean_df["Bathroom"], errors="coerce").fillna(2).astype(int).clip(1, 20)
clean_df["balcony"] = pd.to_numeric(clean_df["Balcony"], errors="coerce").fillna(1).astype(int).clip(0, 10)

# Fill and clean categorical defaults
clean_df["Furnishing"] = clean_df["Furnishing"].fillna("Semi-Furnished").astype(str).str.strip()
clean_df["Transaction"] = clean_df["Transaction"].fillna("Resale").astype(str).str.strip()
clean_df["Ownership"] = clean_df["Ownership"].fillna("Freehold").astype(str).str.strip()
clean_df["facing"] = clean_df["facing"].fillna("East").astype(str).str.strip()

# Group top 50 locations + 'other'
top_50_locs = clean_df["location"].value_counts().head(50).index.tolist()
clean_df["location_grouped"] = clean_df["location"].apply(lambda x: x if x in top_50_locs else "other")

# Outlier trimming based on price-per-sqft
clean_df["price_per_sqft"] = clean_df["price_clean"] / clean_df["carpet_area_sqft"]
q01 = clean_df["price_per_sqft"].quantile(0.01)
q99 = clean_df["price_per_sqft"].quantile(0.99)
clean_df = clean_df[(clean_df["price_per_sqft"] >= q01) & (clean_df["price_per_sqft"] <= q99)].copy()

print(f"Final Clean Dataset Shape: {clean_df.shape[0]:,} rows")

---
## 2.4 Build Scikit-Learn Pipeline & Train Linear Regression

In [ ]:
numeric_features = ["carpet_area_sqft", "floor_num", "bathroom", "balcony"]
categorical_features = ["location_grouped", "Furnishing", "Transaction", "Ownership", "facing"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), categorical_features)
    ],
    remainder="drop"
)

X = clean_df[numeric_features + categorical_features]
y = clean_df["price_clean"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {len(X_train):,}, Test size: {len(X_test):,}")

# Build Linear Regression model with target log transformation
linear_model = Pipeline([
    ("prep", preprocessor),
    ("reg", TransformedTargetRegressor(regressor=LinearRegression(), func=np.log1p, inverse_func=np.expm1))
])

linear_model.fit(X_train, y_train)
print("Linear Regression Pipeline training complete!")

---
## 2.5 Model Evaluation on Test Set

In [ ]:
# Predict on unseen test set
y_pred = linear_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"{'Evaluation Metric':<25} | {'Value':<20}")
print("-" * 48)
print(f"{'Mean Absolute Error (MAE)':<25} | ₹ {mae:,.2f} (₹ {mae/1e5:.2f} Lac)")
print(f"{'Root Mean Squared Error':<25} | ₹ {rmse:,.2f} (₹ {rmse/1e5:.2f} Lac)")
print(f"{'R² Score (Coefficient of Det)':<25} | {r2:.4f}")

In [ ]:
# Predicted vs Actual Scatter Plot
plt.figure(figsize=(9, 6))
plt.scatter(y_test / 1e5, y_pred / 1e5, alpha=0.5, color="#2563eb", edgecolors="none", s=30)
lims = [0, max(y_test.quantile(0.99) / 1e5, y_pred.max() / 1e5)]
plt.plot(lims, lims, "r--", alpha=0.8, linewidth=2, label="Perfect Prediction (y = x)")
plt.title("Predicted vs. Actual Property Price (in Lakhs)", fontsize=14, fontweight="bold")
plt.xlabel("Actual Price (₹ in Lakhs)")
plt.ylabel("Predicted Price (₹ in Lakhs)")
plt.xlim(lims)
plt.ylim(lims)
plt.legend()
plt.tight_layout()
plt.show()

---
## 2.6 Export Model & Location Artifacts

In [ ]:
# Export trained pipeline
joblib.dump(linear_model, "house_price.pkl")
print("Exported model to house_price.pkl")

# Export allowed locations list for frontend and backend
allowed_locations = sorted(clean_df["location_grouped"].unique().tolist())
with open("locations.json", "w", encoding="utf-8") as f:
    json.dump(allowed_locations, f, indent=2)
print(f"Exported {len(allowed_locations)} locations to locations.json")

# Sanity check: Reload and predict single sample
reloaded_model = joblib.load("house_price.pkl")
sample_input = X_test.iloc[[0]]
sample_pred = reloaded_model.predict(sample_input)[0]
print(f"\nSanity Check Prediction:")
print(f"  Input: {sample_input.to_dict(orient='records')[0]}")
print(f"  Predicted Valuation: ₹ {sample_pred:,.2f} (₹ {sample_pred/1e5:.2f} Lac)")